Model Comparison on Breast Cancer Dataset - Decision Trees, Random Forest and Gradient Boosting Machines

Decision Trees

In [1]:
import numpy as np
from collections import Counter

class DecisionTree:
    
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    # gini impurity — measures how mixed a node is
    def gini(self, y):
        counts = Counter(y)
        impurity = 1
        for label in counts:
            prob = counts[label] / len(y)
            impurity -= prob ** 2
        return impurity
    
    # find the best split for a node
    def best_split(self, X, y):
        best_gini = float('inf') # set to worst possible 
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        # try every feature - brute force 
        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            
            # try every unique value as a threshold
            for threshold in thresholds:
                left_mask  = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                y_left  = y[left_mask]
                y_right = y[right_mask]
                
                # skip if one side is empty
                if len(y_left) == 0 or len(y_right) == 0:
                    continue
                
                # weighted gini of this split
                n = len(y)
                gini_split = (len(y_left) / n)  * self.gini(y_left) + \
                             (len(y_right) / n) * self.gini(y_right)
                
                # best split
                if gini_split < best_gini:
                    best_gini      = gini_split
                    best_feature   = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    # build the tree recursively
    def build_tree(self, X, y, depth=0):
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # stopping conditions (if tree reaches max-depth, only one class left, few samples left)
        if depth >= self.max_depth or n_classes == 1 or n_samples < self.min_samples_split:
            # return leaf node — most common class
            return Counter(y).most_common(1)[0][0]
        
        # find best split
        feature, threshold = self.best_split(X, y)
        
        # split data
        left_mask  = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # recursively build left and right subtrees
        left  = self.build_tree(X[left_mask],  y[left_mask],  depth + 1)
        right = self.build_tree(X[right_mask], y[right_mask], depth + 1)
        
        # return node as dictionary
        return {
            'feature'   : feature,
            'threshold' : threshold,
            'left'      : left,
            'right'     : right
        }
    
    # fit() — just builds the tree
    def fit(self, X, y):
        self.root = self.build_tree(X, y)
    
    # predict a single sample
    def predict_one(self, x, node):
        # If leaf node — return class
        if not isinstance(node, dict):
            return node
        
        # go left or right based on threshold
        if x[node['feature']] <= node['threshold']:
            return self.predict_one(x, node['left'])
        else:
            return self.predict_one(x, node['right'])
    
    # predict all samples
    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# load data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train
model = DecisionTree(max_depth=5, min_samples_split=2)
model.fit(X_train, y_train)

# predict
predictions = model.predict(X_test)

# accuracy
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 92.98%


Random Forest

In [1]:
import numpy as np
from collections import Counter

# reusing decision tree code

class DecisionTree:

    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.root = None

    def gini(self, y):
        counts = Counter(y)
        impurity = 1
        for label in counts:
            prob = counts[label] / len(y)
            impurity -= prob ** 2
        return impurity

    def best_split(self, X, y):
        best_gini = float('inf')
        best_feature = None
        best_threshold = None

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                y_left = y[left_mask]
                y_right = y[~left_mask]
                if len(y_left) == 0 or len(y_right) == 0:
                    continue
                n = len(y)
                gini_split = (len(y_left) / n) * self.gini(y_left) + \
                             (len(y_right) / n) * self.gini(y_right)
                if gini_split < best_gini:
                    best_gini = gini_split
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < 2:
            return Counter(y).most_common(1)[0][0]
        feature, threshold = self.best_split(X, y)
        left_mask = X[:, feature] <= threshold
        left = self.build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self.build_tree(X[~left_mask], y[~left_mask], depth + 1)
        return {'feature': feature, 'threshold': threshold, 'left': left, 'right': right}

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node['feature']] <= node['threshold']:
            return self.predict_one(x, node['left'])
        return self.predict_one(x, node['right'])

    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])


class RandomForest:

    def __init__(self, n_trees=100, max_depth=10):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def bootstrap_sample(self, X, y):
        indices = np.random.choice(len(X), len(X), replace=True) # picks rows randomly from the data
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            X_sample, y_sample = self.bootstrap_sample(X, y) # new sample created randomly from rows 
            tree = DecisionTree(max_depth=self.max_depth) # decision tree trained on the new sample
            tree.fit(X_sample, y_sample)
            self.trees.append(tree) # tree stored

    def predict(self, X):
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        # majority vote for each sample
        return np.array([Counter(col).most_common(1)[0][0] for col in all_preds.T])


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForest(n_trees=100, max_depth=10)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 95.61%


Gradient Boosting Machines

In [2]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingClassifier:
    
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators  = n_estimators
        self.learning_rate = learning_rate
        self.max_depth     = max_depth
        self.trees         = []
        self.initial_pred  = None
    
    # sigmoid to convert to probabilities
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def fit(self, X, y):
        #  initial prediction - since we need to have a starting point 
        mean = np.mean(y)
        # log-odds cuz gbm work with log values not probabilities 
        self.initial_pred = np.log(mean / (1 - mean))

        # store initial pred in F
        F = np.full(len(y), self.initial_pred)
        
        for _ in range(self.n_estimators):
            
            # convert F into probability using sigmoid 
            probabilities = self.sigmoid(F)
            # calc residuals (actual - pred)
            residuals     = y - probabilities
            
            # train a tree on residuals = sees how wrong are we
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            
            # update F using the tree's predictions and learning rate
            F = F + self.learning_rate * tree.predict(X)
            
            # save tree
            self.trees.append(tree)
    
    def predict_proba(self, X):
        # start with initial prediction
        F = np.full(X.shape[0], self.initial_pred)
        
        # add each tree's contribution
        for tree in self.trees:
            F = F + self.learning_rate * tree.predict(X)
        
        return self.sigmoid(F)
    
    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Accuracy
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 95.61%


COMPARISON

In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter

#  Train all 3 models with same data split 
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Decision Tree
dt = DecisionTree(max_depth=5)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)

# Random Forest
rf = RandomForest(n_trees=100, max_depth=10)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Gradient Boosting
gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
gbm.fit(X_train, y_train)
gbm_preds = gbm.predict(X_test)
gbm_probs = gbm.predict_proba(X_test)

# Accuracies
dt_acc  = np.mean(dt_preds == y_test) * 100
rf_acc  = np.mean(rf_preds == y_test) * 100
gbm_acc = np.mean(gbm_preds == y_test) * 100

print(f"Decision Tree:     {dt_acc:.2f}%")
print(f"Random Forest:     {rf_acc:.2f}%")
print(f"Gradient Boosting: {gbm_acc:.2f}%")

#  1. Accuracy Bar Chart 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
accs = [dt_acc, rf_acc, gbm_acc]
colors = ['#e74c3c', '#2ecc71', '#3498db']

axes[0].bar(models, accs, color=colors, edgecolor='black')
axes[0].set_ylim(85, 100)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Model Accuracy Comparison')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.3, f'{v:.2f}%', ha='center', fontweight='bold')

#  2. Confusion Matrix Breakdown 
all_preds = [dt_preds, rf_preds, gbm_preds]

cm_data = []
for preds in all_preds:
    cm = confusion_matrix(y_test, preds)
    cm_data.append(cm)

labels = ['True Neg', 'False Pos', 'False Neg', 'True Pos']
dt_vals  = cm_data[0].ravel()
rf_vals  = cm_data[1].ravel()
gbm_vals = cm_data[2].ravel()

x = np.arange(len(labels))
width = 0.25
axes[1].bar(x - width, dt_vals, width, label='Decision Tree', color=colors[0], edgecolor='black')
axes[1].bar(x, rf_vals, width, label='Random Forest', color=colors[1], edgecolor='black')
axes[1].bar(x + width, gbm_vals, width, label='Gradient Boosting', color=colors[2], edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_title('Confusion Matrix Breakdown')
axes[1].legend()

#  3. Where do RF and GBM disagree? 
agree = np.sum(rf_preds == gbm_preds)
disagree = np.sum(rf_preds != gbm_preds)
axes[2].bar(['Agree', 'Disagree'], [agree, disagree], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[2].set_title('RF vs GBM: Agreement on Predictions')
axes[2].set_ylabel('Number of Samples')
for i, v in enumerate([agree, disagree]):
    axes[2].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# 4. GBM probability distribution 
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gbm_probs[y_test == 0], bins=20, alpha=0.7, label='Actual: Malignant (0)', color='#e74c3c')
ax.hist(gbm_probs[y_test == 1], bins=20, alpha=0.7, label='Actual: Benign (1)', color='#2ecc71')
ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold (0.5)')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Count')
ax.set_title('GBM Probability Distribution — How Confident is the Model?')
ax.legend()
plt.tight_layout()
plt.show()

#  5. Show exactly where they differ 
print("\n Where RF and GBM disagree ")
disagree_idx = np.where(rf_preds != gbm_preds)[0]
if len(disagree_idx) == 0:
    print("They agree on every sample! Same accuracy = same predictions here.")
else:
    for idx in disagree_idx:
        print(f"Sample {idx}: True={y_test[idx]}, RF={rf_preds[idx]}, GBM={gbm_preds[idx]}, GBM_prob={gbm_probs[idx]:.4f}")

print("\n Classification Reports")
for name, preds in zip(models, all_preds):
    print(f"\n{name}:")
    print(classification_report(y_test, preds, target_names=['Malignant', 'Benign']))
